# Verifikasi kandidat SLR_09 v0.8.3

Inferensi neural aktual SENA dan NARA pada 150 acuan yang sudah dipakai untuk training/pengembangan. Ini regresi, bukan uji penerimaan baru. NARA dan cutoff 0,95 tetap. Keluaran aktual direkam pada salinan _executed.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
from datetime import datetime, timezone
ROOT = Path(os.environ['SLR09_PACKAGE_ROOT']).resolve()
WORK = Path(os.environ['SLR09_WORK_DIR']).resolve()
SENA = Path(os.environ['SLR09_SENA_DIR']).resolve()
NARA = Path(os.environ['SLR09_NARA_DIR']).resolve()
OUT = WORK / ('joint_regression_v083_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
cmd = [sys.executable, str(ROOT/'code/quality_fix/verify_last_layer_candidate_v083.py'), '--sena-dir', str(SENA), '--nara-dir', str(NARA), '--output', str(OUT)]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
if process.returncode:
    raise RuntimeError('Verifikasi kandidat gagal; pertahankan run_state/error_trace.')
report = json.loads((OUT/'run_state.json').read_text())
assert report['status'] == 'SUCCEEDED' and report['all_seen_joint_regression_checks_met']
assert not report['new_independent_acceptance_executed']
print(json.dumps(report, ensure_ascii=False, indent=2))
